In [8]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [9]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [10]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [11]:
from langchain.agents import create_agent
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.5
)


# create subagents

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [14]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    print(f"subagent_1 = {response['messages'][-1].content}")
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    print(f"subagent_2 = {response['messages'][-1].content}")
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [15]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

subagent_1 = The square root of 456.0 is approximately **21.3541565041**.


In [16]:
from pprint import pprint

pprint(response)
print(f"main = {response['messages'][-1].content}")

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='dd16a046-f6b3-4d60-9af6-b38b2a379da9'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the square root of 456. Let me check which tool I can use here. The available tools are call_subagent_1 for square roots and call_subagent_2 for squares. Since the question is about a square root, I should use call_subagent_1. The parameter needed is x, which is 456 in this case. I need to make sure the input is a number, which it is. So I'll generate the tool call with x=456.\n", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "Okay, the user is asking for the square root of 456. Let me check which tool I can use here. The available tools are call_subagent_1 for square roots and call_subagent_2 for squares. Since the question is about a square root, I should use call_subagent_1. The p